# Gray-Box AIA72 — Mondrian Class-Conditional Conformal v2

Repairs the minority-class failure observed with marginal split conformal. Uses **only** the reserved `conformal_calibration` role and does not read later roles.

In [ ]:

from pathlib import Path
from datetime import datetime, timezone
import hashlib, json, math
import numpy as np
import pandas as pd

# ============================================================
# Gray-Box AIA72 — Class-Conditional (Mondrian) Conformal v2
# ============================================================
# Purpose:
#   Repair the minority-class failure exposed by the original marginal conformal
#   policy-validation diagnostic, while using ONLY the reserved conformal_calibration
#   block to set new thresholds.
#
# Important provenance:
#   - policy_validation outcomes have now been observed for v1 and therefore must NOT
#     be used to fit/select this v2 conformal method.
#   - later Cycle-25 and supplementary-2026 roles remain unopened here.
#   - this notebook does no model fitting/retraining and needs no GPU.

MASTER = Path("/home/abmoses2000/graybox_aia72_master_predictions_v2_20261005/graybox_aia72_master_predictions.csv.gz")
MASTER_SHA = "17300093b88b7c61ecadf5eb28acba0d6d023b340d7740807852dd4352b22b46"

SHARP = Path("/home/abmoses2000/sharp72_predictions_frozen_20261002.csv.gz")
SHARP_SHA = "7368c002a978321cdc67221564f3b7901687badc7aba28ab9df615179d1e017b"

CAL_SELECTION = Path("/home/abmoses2000/multimodal72_calibration_v1_20261005T082136654618Z/selection.json")

OUT = Path("/home/abmoses2000/graybox_aia72_conformal_mondrian_v2_20261005")
if OUT.exists():
    raise SystemExit(f"Refusing to overwrite existing output: {OUT}")
OUT.mkdir(parents=True)

ROLE = "conformal_calibration"
EXPECTED_ROWS = 4168
EXPECTED_POSITIVES = 384
ALPHAS = [0.10, 0.05, 0.20]
PRIMARY_ALPHA = 0.10

def sha256(path):
    h = hashlib.sha256()
    with Path(path).open("rb") as f:
        for chunk in iter(lambda: f.read(4*1024*1024), b""):
            h.update(chunk)
    return h.hexdigest()

def require(cond, msg):
    if not cond:
        raise ValueError(msg)

def finite_qhat(scores, alpha):
    scores = np.asarray(scores, dtype=float)
    n = len(scores)
    require(n > 0, "Empty class-conditional calibration score set")
    k = int(math.ceil((n + 1) * (1 - alpha)))
    k = min(max(k, 1), n)
    return k, float(np.sort(scores)[k - 1])

def set_state(p, q0, q1):
    # score for candidate class 0 is p
    # score for candidate class 1 is 1-p
    include0 = p <= q0
    include1 = (1-p) <= q1
    out = np.empty(len(p), dtype=object)
    out[(~include0)&(~include1)] = "empty"
    out[include0&(~include1)] = "singleton_no_flare"
    out[(~include0)&include1] = "singleton_flare"
    out[include0&include1] = "doubleton"
    return out, include0, include1

require(MASTER.is_file() and SHARP.is_file() and CAL_SELECTION.is_file(), "Missing frozen input")
require(sha256(MASTER) == MASTER_SHA, "Master SHA changed")
require(sha256(SHARP) == SHARP_SHA, "SHARP SHA changed")

selection = json.loads(CAL_SELECTION.read_text())
require(selection["selected_methods"] == {"sharp":"raw","aia":"raw","sharp_aia":"raw"},
        "Probability calibration changed")

master = pd.read_csv(MASTER, low_memory=False)
sharp = pd.read_csv(SHARP, low_memory=False)

a = master.loc[master["role"].astype(str) == ROLE].copy()
s = sharp.loc[sharp["role"].astype(str) == ROLE].copy()

require(len(a) == EXPECTED_ROWS and len(s) == EXPECTED_ROWS, "Conformal support changed")
a["forecast_case_id"] = a["forecast_case_id"].astype(str)
s["forecast_case_id"] = s["forecast_case_id"].astype(str)
require(a["forecast_case_id"].tolist() == s["forecast_case_id"].tolist(), "SHARP/AIA ID order mismatch")

y = pd.to_numeric(a["label"], errors="raise").astype(int).to_numpy()
ys = pd.to_numeric(s["label"], errors="raise").astype(int).to_numpy()
require(np.array_equal(y, ys), "SHARP/AIA labels disagree")
require(int(y.sum()) == EXPECTED_POSITIVES, f"Positive support changed: {int(y.sum())}")

aia_p = pd.to_numeric(a["probability"], errors="raise").to_numpy(float)
sharp_p = pd.to_numeric(s["probability_gru_mean"], errors="raise").to_numpy(float)
fusion_p = 0.5*sharp_p + 0.5*aia_p

for name,p in [("sharp",sharp_p),("aia",aia_p),("sharp_aia",fusion_p)]:
    require(np.isfinite(p).all(), f"{name} has non-finite probabilities")
    require(((p>=0)&(p<=1)).all(), f"{name} probabilities outside [0,1]")

branches = {"sharp":sharp_p, "aia":aia_p, "sharp_aia":fusion_p}
rows = []
diagnostics = []

for branch,p in branches.items():
    # Candidate-class nonconformity scores.
    score0 = p                 # if true class is 0: 1 - (1-p) = p
    score1 = 1.0 - p           # if true class is 1: 1 - p

    cal0 = score0[y == 0]
    cal1 = score1[y == 1]

    for alpha in ALPHAS:
        k0, q0 = finite_qhat(cal0, alpha)
        k1, q1 = finite_qhat(cal1, alpha)

        rows.append({
            "branch": branch,
            "alpha": alpha,
            "nominal_class_conditional_coverage": 1-alpha,
            "class0_n": len(cal0),
            "class1_n": len(cal1),
            "class0_k": k0,
            "class1_k": k1,
            "qhat_class0": q0,
            "qhat_class1": q1,
            "min_prob_for_class1_membership": 1-q1,
            "max_prob_for_class0_membership": q0,
            "method": "class_conditional_mondrian_split_conformal",
        })

        state, include0, include1 = set_state(p, q0, q1)
        contains_true = np.where(y == 0, include0, include1)
        diagnostics.append({
            "branch": branch,
            "alpha": alpha,
            "overall_empirical_coverage_on_calibration": float(contains_true.mean()),
            "class0_empirical_coverage_on_calibration": float(include0[y==0].mean()),
            "class1_empirical_coverage_on_calibration": float(include1[y==1].mean()),
            "empty_set_rate": float((state=="empty").mean()),
            "singleton_no_flare_rate": float((state=="singleton_no_flare").mean()),
            "singleton_flare_rate": float((state=="singleton_flare").mean()),
            "doubleton_rate": float((state=="doubleton").mean()),
        })

thresholds = pd.DataFrame(rows)
diag = pd.DataFrame(diagnostics)

threshold_path = OUT / "mondrian_thresholds.csv"
diag_path = OUT / "calibration_diagnostics.csv"
thresholds.to_csv(threshold_path, index=False)
diag.to_csv(diag_path, index=False)

primary = thresholds.loc[np.isclose(thresholds["alpha"], PRIMARY_ALPHA)].copy()
primary_diag = diag.loc[np.isclose(diag["alpha"], PRIMARY_ALPHA)].copy()

protocol = {
    "status": "MONDRIAN_CLASS_CONDITIONAL_CONFORMAL_V2_FROZEN",
    "scientific_acceptance": False,
    "created_utc": datetime.now(timezone.utc).isoformat(),
    "horizon_hours": 72,
    "source_role": ROLE,
    "rows": EXPECTED_ROWS,
    "positives": EXPECTED_POSITIVES,
    "negatives": int(EXPECTED_ROWS-EXPECTED_POSITIVES),
    "method": "class_conditional_mondrian_split_conformal",
    "score_class0": "p(flare)",
    "score_class1": "1 - p(flare)",
    "primary_alpha": PRIMARY_ALPHA,
    "sensitivity_alphas": [0.05,0.20],
    "probability_calibration": {"sharp":"raw_identity","aia":"raw_identity","sharp_aia":"raw_identity"},
    "fusion_weights": {"sharp":0.5,"aia":0.5},
    "why_v2_exists": (
        "The earlier marginal conformal v1 achieved high marginal coverage on policy_validation "
        "but near-zero positive-class coverage for AIA and fusion. This v2 is a methodological "
        "repair for class imbalance: thresholds are calibrated separately by true class using "
        "only the reserved conformal_calibration block."
    ),
    "policy_validation_used_to_fit_v2": False,
    "later_roles_read": False,
    "future_evaluation_rule": (
        "Do not tune v2 on the already-seen policy_validation outcomes. After freezing downstream "
        "v2 policy rules, evaluate them on still-unopened retrospective_cycle25 and supplementary_2026 "
        "as the clean future-period assessment."
    ),
    "source_sha256": {
        "master_predictions": MASTER_SHA,
        "sharp_predictions": SHARP_SHA,
        "probability_selection": sha256(CAL_SELECTION),
    },
    "output_sha256": {
        threshold_path.name: sha256(threshold_path),
        diag_path.name: sha256(diag_path),
    }
}
(OUT/"protocol.json").write_text(json.dumps(protocol, indent=2)+"\n")

print("===== MONDRIAN CONFORMAL V2 FROZEN =====")
print("rows:", EXPECTED_ROWS)
print("positives:", EXPECTED_POSITIVES)
print("policy_validation used to fit v2: False")
print("later roles read: False")
print("\nPrimary alpha=0.10 thresholds:")
print(primary.to_string(index=False))
print("\nPrimary alpha=0.10 calibration diagnostics:")
print(primary_diag.to_string(index=False))
print("\nProtocol:")
print(json.dumps(protocol, indent=2))
print("\nSaved:", OUT/"protocol.json")
print("Saved:", threshold_path)
print("Saved:", diag_path)
